# Lab 3 — เปิดตัว Microduck ใน MuJoCo แล้วดูมันล้ม

**EP01 · Physical AI คืออะไร** · คลิปก่อนหน้า: Lab 2 Python, Git และ Colab

ใน Lab 1 เราขับ Microduck ที่มี policy คุมอยู่ Lab นี้เราจะเปิด **model ของมันเอง** ใน MuJoCo
ดูว่ามันมี joint, motor และ sensor อะไร แล้วลองสั่งให้มันยืนท่าเดิมเฉยๆ โดยไม่มี policy

> ✏️ **Starter notebook — ฉบับให้ผู้เรียนลงมือเขียน**
>
> โค้ดส่วนใหญ่เตรียมไว้ให้แล้ว เขียนเพิ่มเฉพาะ **4 จุดที่มี `✏️ TODO`** (แทนที่ `___` หรือ `...` ด้วยโค้ดของคุณ) ทุกจุดบอกเวลาในวิดีโอที่อธิบายโค้ดตรงนั้น (▶ m:ss) — ดูคลิปก่อน แล้วลองเขียนเองโดยไม่เปิดเฉลย
>
> รันทุก cell ตามลำดับจากบนลงล่าง ถ้า cell ไหนยังเติมไม่ครบจะเจอ `NameError` หรือ `TypeError` เป็นเรื่องปกติ
>
> ติดตรงไหน เปิดเฉลยใน **`ep01-lab3-meet-microduck.ipynb`** (โค้ดชุดเดียวกับในวิดีโอ)

## เตรียมตัว

- ติดตั้ง **MuJoCo 3.17.0** (ตรึงเวอร์ชันไว้ เพื่อให้ผลตรงกับวิดีโอ) · NumPy, Pillow, matplotlib มีใน Colab อยู่แล้ว
- บน Colab ไม่มีหน้าจอ จึงตั้ง `MUJOCO_GL=egl` ให้ MuJoCo วาดภาพแบบ off-screen
- model ของ Microduck มาจาก repo `pollen-robotics/microduck_rl` ที่ commit `bae39f5` (ตรึงไว้ทั้งเทอม)
- ไฟล์ MJCF เป็น Apache-2.0 · ไฟล์ 3 มิติ (`*.stl`) © Pollen Robotics, **CC BY-NC-SA 4.0**
- วิธีติดตั้งและใช้ Colab อยู่ใน Lab 2

In [ ]:
%pip install -q mujoco==3.17.0

import os, sys
if "google.colab" in sys.modules:
    os.environ["MUJOCO_GL"] = "egl"   # Colab has no screen: render off-screen
import mujoco
print(mujoco.__version__)

!git clone -q https://github.com/pollen-robotics/microduck_rl.git
!git -C microduck_rl checkout -q bae39f5900b318f7e87fc485c30a31213cfa14eb
SCENE = "microduck_rl/src/mjlab_microduck/robot/microduck/scene.xml"

## เปิด model: Microduck มีอะไรบ้าง

- **model** (`MjModel`) = สิ่งที่ไม่เปลี่ยน: รูปร่าง มวล joint motor sensor
- **data** (`MjData`) = สิ่งที่เปลี่ยนทุก step: มุม ความเร็ว เวลา ค่าที่ sensor วัดได้

In [ ]:
# ✏️ TODO 1 (▶ วิดีโอ 1:34): โหลด model จากไฟล์ SCENE
model = ___
data = mujoco.MjData(model)
print("joints:", model.njnt, "| motors:", model.nu, "| sensors:", model.nsensor)

In [ ]:
for i in range(model.njnt):
    print(i, model.joint(i).name)

In [ ]:
for i in range(model.nsensor):
    print(model.sensor(i).name, "->", model.sensor_dim[i], "numbers")

## ทดลอง: ยืนท่าเดิมเฉยๆ แล้วปล่อยมือ

model มีท่าที่ตั้งไว้ชื่อ **STAND** · เราจะตั้งหุ่นในท่านี้ สั่งทุก motor ให้ค้างมุมเดิม แล้วปล่อยให้ฟิสิกส์ทำงาน 1.5 วินาที

In [ ]:
import numpy as np
from PIL import Image
renderer = mujoco.Renderer(model, height=360, width=640)
cam = mujoco.MjvCamera()
cam.lookat[:] = [0, 0, 0.08]
cam.distance, cam.azimuth, cam.elevation = 0.6, 135, -15

def snapshot(data):
    renderer.update_scene(data, camera=cam)
    return renderer.render()

# ✏️ TODO 2 (▶ วิดีโอ 3:29): ตั้งหุ่นในท่า STAND (keyframe)

mujoco.mj_resetDataKeyframe(model, data, ___)
mujoco.mj_forward(model, data)
Image.fromarray(snapshot(data))

In [ ]:
# ✏️ TODO 3 (▶ วิดีโอ 3:52): เดินโลกจำลองไปหนึ่ง step
times, heights, frames = [], [], []
while data.time < 1.5:
    ___
    times.append(data.time)
    heights.append(data.qpos[2])          # trunk height (m)
    if len(times) % 150 == 0 and len(frames) < 4:
        frames.append(snapshot(data))     # a picture every 0.3 s

heights = np.array(heights)
times = np.array(times)
print("trunk below 9 cm at", times[heights < 0.09][0].round(2), "s")
Image.fromarray(np.hstack(frames))

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 3))
plt.plot(times, heights * 100, label="hold STAND (XML servos)")
plt.xlabel("time (s)"); plt.ylabel("trunk height (cm)")
plt.legend(); plt.show()

## ลองเอง: ถ้า servo แข็งขึ้นล่ะ?

**kp** คือความแข็งของ servo: มุมหลุดจากเป้าหมายเท่าเดิม ถ้า kp มาก servo จะดึงกลับแรงขึ้น (เรียนจริงจังใน EP05)
ใน XML ตั้งไว้ 0.55 · ลองเปลี่ยนเป็น 2.0 ซึ่งเป็นค่าที่ `microduck_rl` ใช้ตอนหุ่นยืนรอ (standby)

> ⚠️ นี่เป็น servo แบบง่ายของ MuJoCo ไม่ใช่ model ของ servo ที่ใช้ตอน train policy จริง

In [ ]:
# ✏️ TODO 4 (▶ วิดีโอ 5:42): เปลี่ยน kp ของทุก servo เป็น 2.0 (ทั้งสองที่)
model.actuator_gainprm[:, 0] = ___      # stiffer servos: kp 0.55 -> 2.0
model.actuator_biasprm[:, 1] = ___

mujoco.mj_resetDataKeyframe(model, data, model.key("STAND").id)
heights2 = []
while data.time < 1.5:
    mujoco.mj_step(model, data)
    heights2.append(data.qpos[2])
print("lowest trunk height:", round(min(heights2) * 100, 1), "cm")

plt.figure(figsize=(7, 3))
plt.plot(times, heights * 100, label="XML servos (kp 0.55)")
plt.plot(times, np.array(heights2) * 100, label="stiffer servos (kp 2.0)")
plt.xlabel("time (s)"); plt.ylabel("trunk height (cm)")
plt.legend(); plt.show()
Image.fromarray(snapshot(data))

## สรุป + งาน

- Microduck ใน model มี **14 joint ที่มี motor** (ขาละ 5, คอและหัว 4) และ sensor ชุด **IMU** · ไม่มี sensor ไหนบอกตำแหน่งเท้าตรงๆ
- ค้างท่ายืนด้วย servo ตาม XML: ลำตัวต่ำกว่า 9 cm ที่ **0.86 s** · servo แข็งขึ้นยืนได้ แต่ยังไม่ได้ทรงตัวเอง

**งานของ Class 1:** เขียนสรุปหนึ่งหน้าว่า *Microduck ต้องรู้อะไรและทำอะไรบ้างจึงจะเดินได้* ใช้รายชื่อ joint และ sensor จาก Lab นี้ และสิ่งที่เห็นใน Lab 1

**คำถามค้าง:** หุ่นจะรู้ได้อย่างไรว่าเท้าของมันอยู่ตรงไหน?

**ถัดไป:** สรุป EP01